# Museum Heist -- PG-2

This is the **Reinforcement Learning** project developed during Erasmus exchange at the University of Milan.
The project focuses on stochastic policies and policy gradient, and it's scientific objective
is to investigate the challenges related to strategic environments that require the use of stochastic policies.

The information below is just a small debrief of the task.
For more detailed task description please refer to the **README.md** file in this repository.

## Problem Description

A thief is trying to steal a specific painting from an art museum.

A security camera is located in the correspondence of each exhibition room,
but the security guard in the control room can only watch one video feed at a time.
Once per minute, an algorithm selects the next location to show on the screen.

Simultaneously, the thief **can move to a nearby room** or **stay in the same room**.
The thief starts from an unknown initial room, has to reach an unknown target room
(the one with the painting they want to steal) and go back to the initial room in order to escape.
The thief has a map of the museum and is also in radio contact with a hacker who knows the current active security camera.

The episode ends if:
- the surveillance algorithm selects the room currently occupied by the thief,
- thief escapes with the painting (select the target room after the painting has been stolen, but the thief has already escaped).

In [471]:
# imports and metaconstraints
import gymnasium as gym
from gymnasium import spaces
from enum import Enum, auto
import networkx as nx
import numpy as np

from utils import map_museum_to_graph

class SpawnStrategy(Enum):
    FIXED = auto()      # Always bottom-left to top-right
    DIVIDED = auto()    # Random, but Thief on Left, Painting on Right
    RANDOM = auto()     # Completely random everywhere

## Environment and Thief

what we have, how it's modeled, motivate choice wrt state/action structure and difficulty,

we allow to fix the position of start and painting to see what happens

we also allow to "divide" the museum into left and right side (enforcing that spawn point is somewhere always on the left side of the museum
and painting position is always on the right side of the museum to see what happens and how the agent learns)

Model the environment in a way, that graph vertices are rooms and edges are doors between rooms.
We want to design environment in a grid-way style, that is:

-- graph vertices are organized into n x m grid

-- each vertex is connected to at most 4 other vertices

-- networkx grid_2d_graph makes all the work for us.

we pass into env function the map of the museum, so that we can easily create other museum topologies
with walls, etc.


In [472]:
# Even though the security camera (agent) is stateless, the environment itself needs to track everything to run the simulation and provide rewards.
# Agent: Security Camera (stateless)
# Adversary: Thief (stateful, shortest-path logic)

class MuseumHeistEnv(gym.Env):
    def __init__(self, museum_map, layout: SpawnStrategy = SpawnStrategy.RANDOM, b=2.0):
        # Initialize the learning environment.
        super().__init__()

        # Environmental metadata.
        self.beta = b
        self.museum_map = museum_map
        self.height = len(museum_map)
        self.width = len(museum_map[0])
        self.museum_layout = layout

        # Museum 2D grid topology, where graph nodes are denoted by (x, y).
        # We map each graph node (x, y) to integer from 0 to N-1 and vice versa to represent room numbers.
        self.museum_graph, self.num_rooms = map_museum_to_graph(museum_map)
        self.node_to_idx = {n: i for i, n in enumerate(self.museum_graph.nodes())}
        self.idx_to_node = {i: n for n, i in self.node_to_idx.items()}

        # Gym spaces -- Action (room selection) & Observation space for the agent.
        # Since policy is stochastic, the observation always returns 0 anyway (stateless).
        self.action_space = spaces.Discrete(self.num_rooms)
        self.observation_space = spaces.Discrete(1)

        # Internal state variables for cost calculation and episode finish.
        # n_counts represents for each room, the number of rounds since i-th room was last selected by surveillance algorithm.
        # Cost to enter room i-th is 1 + self.beta * 2 ^ { -(n_counts[i] - 1) }
        self.n_counts = np.ones(self.num_rooms)
        self.thief_pos = None
        self.painting_pos = None
        self.start_pos = None
        self.has_painting = False

    def reset(self, seed=None, options=None):
        # Restart the environment after episode finishes.
        super().reset(seed=seed)

        if self.museum_layout == SpawnStrategy.FIXED:
            # Fix the scenario: e.g., Enter bottom-left, steal from top-right
            self.start_pos = self.node_to_idx[(0, self.height-1)]
            self.painting_pos = self.node_to_idx[(self.width-1, 0)]

        elif self.museum_layout == SpawnStrategy.DIVIDED:
            # Filter valid rooms into "Left" and "Right" sides (node[0] is the x-coordinate)
            left_side_rooms = [node for node in self.museum_graph.nodes() if node[0] < self.width // 2]
            right_side_rooms = [node for node in self.museum_graph.nodes() if node[0] > self.width // 2]
            start_node = left_side_rooms[self.np_random.choice(len(left_side_rooms))]
            target_node = right_side_rooms[self.np_random.choice(len(right_side_rooms))]
            self.start_pos = self.node_to_idx[start_node]
            self.painting_pos = self.node_to_idx[target_node]

        else: # SpawnStrategy.RANDOM
            # Choose Thief starting position and painting position at random at the beginning of each episode.
            # We choose two at once with replace=False to avoid duplicating the same position.
            positions = self.np_random.choice(self.num_rooms, 2, replace=False)
            self.start_pos = positions[0]
            self.painting_pos = positions[1]

        # Initialize episode metadata at each reset and return (nothing, as we're stateless).
        self.thief_pos = self.start_pos
        self.has_painting = False
        self.n_counts = np.ones(self.num_rooms)
        return 0, {}

    def _calculate_next_thief_step(self, active_camera):
        # Perform one step of Thief in the museum topology.
        # Since Thief next move is state/environmental-dependant, based on dynamic cost,
        # we calculate the shortest path inside it and return next step on Thief's path.

        # Our museum graph has to be a DIRECTED graph, since the cost of entering i-th room
        # from j-th room can be different from entering j-th room from i-th room, since n_counts[i] != n_counts[j].
        # m_graph is the graph with current state/costs of entering each room.
        # For every edge in the initial Museum Map, we calculate both edges of entering u and v.
        m_graph = nx.DiGraph()
        for u, v in self.museum_graph.edges():
            u_idx, v_idx = self.node_to_idx[u], self.node_to_idx[v]

            # First direction
            cost_v = 1.0 + self.beta * (2.0 ** -(self.n_counts[v_idx] - 1))
            m_graph.add_edge(u_idx, v_idx, weight=cost_v)

            # Second direction
            cost_u = 1.0 + self.beta * (2.0 ** -(self.n_counts[u_idx] - 1))
            m_graph.add_edge(v_idx, u_idx, weight=cost_u)

        # Hacker intervention (remove the possibility to move node currently surveilled).
        if active_camera in m_graph.nodes and active_camera != self.thief_pos:
            m_graph.remove_node(active_camera)

        current_target_room = self.start_pos if self.has_painting else self.painting_pos

        try:
            path = nx.shortest_path(m_graph, source=self.thief_pos, target=current_target_room, weight='weight')
            # If the Thief is not standing in the node that source == target.
            # Take the next step on the shortest path, path[0] is the source.
            return path[1] if len(path) > 1 else self.thief_pos
        except (nx.NetworkXNoPath, nx.NodeNotFound):
            # If the only move possibility is blocked by the camera, we enforce the Thief to stay in the same room.
            # If NodeNotFound (removed, as Hacker told us that camera is currently in the room we want to reach), also stay in the same place.
            return self.thief_pos

    def step(self, a):
        # Perform one step of the environment (Thief movement, surveillance action).
        # Return is a tuple (observation -- 0 since stateless, reward, terminate, truncated, info)

        # 1. Update surveillance counters
        self.n_counts += 1
        self.n_counts[a] = 1

        # 2. Check whether Agent caught the Thief, reward if so.
        # Chosen room is the action fed into function (action taken by the agent).
        if a == self.thief_pos:
            return 0, 1.0, True, False, {"reason": "thief caught"}

        # 3. If Thief was not caught, move him to next position with already updated n_counts.
        # As Thief is in communication with a Hacker, that knows the currently active camera,
        # communicate it to the Thief, so he doesn't enter that room.
        self.thief_pos = self._calculate_next_thief_step(active_camera=a)

        # 4. Verify whether Thief is in the painting's room.
        if not self.has_painting and self.thief_pos == self.painting_pos:
            self.has_painting = True

        # 5. Check whether Agent let the Thief escape with the painting, reward accordingly.
        if self.has_painting and self.thief_pos == self.start_pos:
            return 0, -1.0, True, False, {"reason": "thief escaped"}

        # Episode continues
        return 0, 0.0, False, False, {}

## Agent -- The Security Camera (Stateless Softmax Policy Gradient)

some descrption

In [473]:
class StatelessSoftmaxAgent:
    def __init__(self, num_rooms, t=1.0, learning_rate=0.05):
        # Policy hyperparameters
        self.theta = np.zeros(num_rooms)
        self.tau = t
        self.lr = learning_rate
        self.num_rooms = num_rooms

    def get_probabilities(self):
        # Calculations as in the lecture notes related Finite Horizon PG.
        logits = self.theta / self.tau
        logits -= np.max(logits)    # Numerical stability for Softmax
        exp_logits = np.exp(logits)
        return exp_logits / np.sum(exp_logits)

    def select_action(self):
        # Actions are sampled based on pi_theta(a) ~ exp(theta_a / tau)
        probs = self.get_probabilities()
        return np.random.choice(self.num_rooms, p=probs)

    def update(self, traj, use_baseline=False, g=0.99):
        # Since we're stateless, given input trajectory is just a list of (action, reward) tuples.
        probs = self.get_probabilities()

        # The gradient update for θ will rely solely on the score function (log derivative) of the softmax policy.
        # We use gradients multiplied by the Return (or expected Return) to update θ via gradient ascent.

        # Compute returns for each (future) step with discount factor gamma.
        returns = []
        total_future_reward = 0
        for _, r in reversed(traj):
            total_future_reward = r + g * total_future_reward
            returns.append(total_future_reward)

        # We introduce a baseline to REINFORCE update, as possible input parameter, to test and compare the performance.
        returns = np.array(returns)[::-1]
        if use_baseline: returns = returns - np.mean(returns)

        # Update theta parameters using REINFORCE (Policy Gradient Theorem).
        # Compute gradient of log pi and then the gradient ascent.
        for t, (a, _) in enumerate(traj):
            grad_log_pi = -probs.copy() / self.tau
            grad_log_pi[a] += 1.0 / self.tau
            self.theta = self.theta + self.lr * (grad_log_pi * returns[t])

## Training the agent

some description

In [474]:
# Training metadata
custom_map = [
    "OOOOO",
    "OOWOO",
    "OOOOO",
    "OOWOO",
    "OOWOO",
]

episodes = 5000
gamma = 0.99
beta = 2.0
museum_layout = SpawnStrategy.DIVIDED
temperature = 1.0
lr = 0.05
baseline = False

env = MuseumHeistEnv(museum_map=custom_map, layout=museum_layout, b=beta)
agent = StatelessSoftmaxAgent(num_rooms=env.num_rooms, t=temperature, learning_rate=lr)

In [475]:
# print("Starting training...")
# for ep in range(episodes):
#     obs, _ = env.reset()
#     done = False
#     trajectory = []
#
#     while not done:
#         action = agent.select_action()
#         _, reward, terminated, truncated, _ = env.step(action)
#         trajectory.append((action, reward))
#         done = terminated or truncated
#
#     # Standard REINFORCE update after episode ends
#     agent.update(trajectory, gamma)
#
#     if (ep + 1) % 500 == 0:
#         print(f"Episode {ep + 1} completed.")

In [476]:
# win_rate_history = []
# wins = 0
#
# for ep in range(episodes):
#     obs, _ = env.reset()
#     done = False
#     trajectory = []
#
#     while not done:
#         action = agent.select_action()
#         next_obs, reward, terminated, truncated, info = env.step(action)
#
#         trajectory.append((action, reward))
#         done = terminated or truncated
#
#         if reward > 0:
#             wins += 1
#
#     # Train at the end of the episode
#     agent.update(trajectory, use_baseline=baseline, g=gamma)
#
#     # Track performance
#     if (ep + 1) % 100 == 0:
#         win_rate = wins / 100.0
#         win_rate_history.append(win_rate)
#         print(f"Episode {ep+1} | Win Rate (last 100): {win_rate*100:.1f}%")
#         wins = 0
#
# # (Task 5 prep) Print the final learned distribution
# final_probs = agent.get_probabilities()
# print("\nFinal Policy Action Distribution (with -1 for walls):")
#
# # 1. Create a full 5x5 grid initialized entirely with -1.0
# full_grid = np.full((env.height, env.width), -1.0)
#
# # 2. Iterate through the agent's 21 probabilities
# for idx, prob in enumerate(final_probs):
#     # Get the actual x, y coordinates for this action index
#     x, y = env.idx_to_node[idx]
#
#     # 3. Overwrite the -1 with the actual probability for valid rooms
#     full_grid[y, x] = prob
#
# # Print the final reconstructed grid
# print(np.round(full_grid, 3))

Episode 4500 | Win Rate (last 100): 69.0%
Episode 4600 | Win Rate (last 100): 73.0%
Episode 4700 | Win Rate (last 100): 72.0%
Episode 4800 | Win Rate (last 100): 65.0%
Episode 4900 | Win Rate (last 100): 73.0%
Episode 5000 | Win Rate (last 100): 75.0%

Final Policy Action Distribution (with -1 for walls):
[[ 0.001  0.006  0.436  0.005  0.001]
 [ 0.001  0.002 -1.     0.002  0.002]
 [ 0.002  0.241  0.002  0.289  0.001]
 [ 0.002  0.001 -1.     0.002  0.   ]
 [ 0.     0.002 -1.     0.002  0.   ]]


# Evaluations

trzeba samemu wymyslec jak go zewaluowac, duzo bedzie tu zmiennych i mozliwosci, raport bedzie dlugi prawdopodobnie